
# 01b · Market Data — Ocupación y RevPAR (con inventario)
**Objetivo:** Unir las métricas mensuales con el **inventario de noches disponibles** para calcular **Ocupación%** y **RevPAR** y exportar un dataset listo para BI.


In [ ]:


import os ,pandas as pd ,numpy as np 

CWD =os .getcwd ()
PROJ =os .path .abspath (os .path .join (CWD ,".."))if os .path .basename (CWD ).lower ()=="notebooks"else CWD 
DATA_PROC =os .path .join (PROJ ,"data","processed")

metrics_path =os .path .join (DATA_PROC ,"metrics_monthly_all.csv")
inventory_path =os .path .join (PROJ ,"inventory_available_nights_template.csv")

metrics_path ,inventory_path 


In [ ]:


assert os .path .exists (metrics_path ),"Falta data/processed/metrics_monthly_all.csv (ejecuta 01_cleaning_and_eda primero)"
metrics =pd .read_csv (metrics_path ,parse_dates =["month"])


inv =pd .read_csv (inventory_path )
if "month"in inv .columns :
    inv ["month"]=pd .to_datetime (inv ["month"])
else :
    raise ValueError ("El inventario debe tener columna 'month' (YYYY-MM-01)")

metrics .head (3 ),inv .head (3 )



## 3) Unión y cálculo de Ocupación% y RevPAR
- **Ocupación%** = `noches_reservadas / available_nights * 100`
- **RevPAR** = `revenue_total / available_nights`


In [ ]:

df =metrics .merge (inv [["month","available_nights"]],on ="month",how ="left")


df ["available_nights"]=pd .to_numeric (df ["available_nights"],errors ="coerce")
df ["available_nights_filled"]=df ["available_nights"].fillna (method ="ffill").fillna (method ="bfill")

df ["occupancy_pct"]=np .where (df ["available_nights_filled"]>0 ,
df ["noches_reservadas"]/df ["available_nights_filled"]*100 ,np .nan ).round (2 )
df ["revpar"]=np .where (df ["available_nights_filled"]>0 ,
df ["revenue_total"]/df ["available_nights_filled"],np .nan ).round (2 )

df_out =df .copy ()
df_out_path =os .path .join (DATA_PROC ,"metrics_monthly_with_occ_revpar.csv")
os .makedirs (DATA_PROC ,exist_ok =True )
df_out .to_csv (df_out_path ,index =False )

df_out_path ,df_out .head (3 )
